# TrustLap: GPU benchmark (M1)

Measures how many simulated car-steps per second the environment runs on this GPU, on this
runtime's CPU, and as a plain NumPy loop. Writes `bench.json` and downloads it.

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → **Run all**.
Takes about 10 minutes. Everything here just calls the `trustlap` package; no logic lives in the notebook.

In [ ]:
# 1. Check the GPU is visible.
!nvidia-smi --query-gpu=name,memory.total --format=csv
import jax

print("jax", jax.__version__, jax.devices())
assert jax.devices()[0].platform == "gpu", (
    "JAX can't see a GPU. Runtime > Change runtime type > T4 GPU, then run all again. "
    "If it still fails: !pip install -U 'jax[cuda12]' then Runtime > Restart session."
)

In [ ]:
# 2. Get the code and install the package (keeps Colab's own CUDA build of JAX).
!rm -rf trustlap && git clone -q --depth 1 https://github.com/hitenjain22/trustlap.git
!pip install -q -e trustlap/sim
!cd trustlap && git log -1 --format='commit %h %s'

In [ ]:
# 3. Show the installed versions (bench.json records them too).
!pip list 2>/dev/null | grep -E '^(jax|jaxlib|jax-cuda12-plugin|flax|optax|distrax|chex|numpy) '

In [ ]:
# 4. Run the benchmark (~10 min). Saves bench.json after every measurement.
# GPU JAX: 1 to 65,536 cars; CPU JAX: 1, 64, 1,024, 65,536; NumPy loop: up to 1,024.
!cd trustlap/sim && python -m trustlap.bench --out bench.json

In [ ]:
# 5. Download bench.json, then send it to Claude Code (or put it in sim/runs/bench.json).
from google.colab import files

files.download("trustlap/sim/bench.json")